# 11주 · 평균차·비율·분산 추정

공학통계학 CE202
슬라이드의 개념·숫자·풀이 순서와 연결한 **새 병행 실습**. 원본 Colab 복원본이 아닌 신규 자료

**목표**: 자료·확률·분포·추정·검정의 원리 이해와 통계적 해석 심화

학습 주제: 추정 (3) / 추정 (4)

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE202-statistics-for-engineers/week11/CE202_W11.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE202-statistics-for-engineers/week11)

질문·가정 확인 → 손계산·예측 → 계산·시각화 → 한 조건 변경 → 통계적 해석·한계 설명

셀 실행: Shift+Enter. 아래 코드의 None은 직접 채울 항목. 원본과 변경 사본의 결과를 구분

핵심 실습: 손계산·원리와 계산식의 연결·조건 변경·해석. AI 활용은 개념 학습 후 선택 활동

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE202-statistics-for-engineers/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)

## 0. 실행 환경

필요한 패키지의 최소 버전 확인. 기본 코드와 보조 함수는 이번 주 실습에 필요한 도구. 세부 구현 암기 불필요

In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.16", "matplotlib": "3.10", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math, tempfile, base64, zipfile, io
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
WORK = Path(tempfile.mkdtemp(prefix="ce202-"))
rng = np.random.default_rng(2027)

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 수치 자료와 최소 표본수 확인 필요")
    return x

def probabilities(values):
    p = vector(values)
    if np.any(p < 0) or not np.isclose(p.sum(), 1):
        raise ValueError("확률의 비음수성·합계 1 확인 필요")
    return p

def mean_ci(values, confidence=0.95):
    x = vector(values, 2)
    if not 0 < confidence < 1:
        raise ValueError("신뢰수준은 0과 1 사이")
    se = x.std(ddof=1) / np.sqrt(len(x))
    q = stats.t.ppf((1 + confidence) / 2, df=len(x)-1)
    return {"n": len(x), "mean": x.mean(), "se": se,
            "df": len(x)-1, "low": x.mean()-q*se, "high": x.mean()+q*se}

def wilson_interval(successes, n, confidence=0.95):
    if not isinstance(n, (int, np.integer)) or n <= 0 or not 0 <= successes <= n:
        raise ValueError("0 ≤ 성공수 ≤ 양의 정수 표본수 확인 필요")
    if int(successes) != successes or not 0 < confidence < 1:
        raise ValueError("정수 성공수·신뢰수준 확인 필요")
    z = stats.norm.ppf((1+confidence)/2)
    phat = successes / n
    den = 1 + z*z/n
    center = (phat + z*z/(2*n)) / den
    half = z / den * np.sqrt(phat*(1-phat)/n + z*z/(4*n*n))
    return center-half, center+half

def anova_summary(groups):
    arrays = [vector(g, 2) for g in groups]
    if len(arrays) < 2:
        raise ValueError("두 집단 이상 필요")
    n = np.array([len(g) for g in arrays]); means = np.array([g.mean() for g in arrays])
    grand = np.concatenate(arrays).mean()
    ss_between = np.sum(n*(means-grand)**2)
    ss_within = sum(np.sum((g-g.mean())**2) for g in arrays)
    df_between = len(arrays)-1; df_within = n.sum()-len(arrays)
    if ss_within <= 0:
        raise ValueError("집단 내 변동이 0이면 일반적인 F 계산 불가")
    F = (ss_between/df_between) / (ss_within/df_within)
    return {"ss_between": ss_between, "ss_within": ss_within,
            "df_between": df_between, "df_within": df_within,
            "F": F, "p": stats.f.sf(F, df_between, df_within)}

def regression(x, y):
    x = vector(x, 3); y = vector(y, 3)
    if len(x) != len(y) or np.ptp(x) == 0:
        raise ValueError("같은 길이·서로 다른 x 값 확인 필요")
    fit = stats.linregress(x, y)
    fitted = fit.intercept + fit.slope*x
    residual = y-fitted
    sse = np.sum(residual**2); sst = np.sum((y-y.mean())**2)
    if sst <= 0:
        raise ValueError("y 변동이 0이면 일반적인 R² 해석 불가")
    return {"slope": fit.slope, "intercept": fit.intercept, "r2": 1-sse/sst,
            "sse": sse, "mse_residual": sse/(len(x)-2), "mse_prediction": sse/len(x),
            "p": fit.pvalue, "stderr": fit.stderr, "fitted": fitted, "residual": residual}


## S01. 원본 교육 전후의 대응 차이

**슬라이드 연결**: CE202_W11a_Inference3.pptx p11, 12, 20, 22, 23, 24

**자료 구분**: 원본 예제·수치 기반

**먼저 예측**: 대응 차이의 분산과 두 독립집단 분산의 합이 같은가?

계산 전에 분모·단위·조건을 한 줄로 기록

In [ ]:
before=np.array([75, 70, 88, 85, 95, 65, 87, 70, 95, 85]); after=np.array([80, 80, 90, 90, 90, 60, 90, 85, 90, 93]); differences=after-before
paired_ci=mean_ci(differences)
print("후−전 차이:",differences,"\n",paired_ci)
assert np.isclose(differences.mean(),3.3) and paired_ci["low"]<0<paired_ci["high"]
# 원본 필터 수명 예제의 대표본 정규근사: 평균차6, n각50, 표본표준편차6/4
se_diff=np.sqrt(6**2/50+4**2/50)
print("원본 대표본 z근사 구간:",6-stats.norm.isf(.025)*se_diff,6+stats.norm.isf(.025)*se_diff)

**조건 변경**: 교육 후 순서만 섞은 사본에서 짝의 의미·표준오차 변화 확인

위 코드를 복사한 셀에서 한 조건씩 변경. 원본 셀은 보존

**선택 심화 · AI 검토**: 동일 참가자10명·후−전 부호·대응t구간·비교군 유무 명시

**결과 검토**: 대응 전후 차이만으로 교육 프로그램의 인과효과 확정 금지

예측·손계산·변경 결과·개념 설명·적용 한계 기록

In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


## S02. 원본 100개 중 2개 불량: Wald와 Wilson

**슬라이드 연결**: CE202_W11a_Inference3.pptx p32, 34

**자료 구분**: 원본 예제·수치 기반

**먼저 예측**: 불량률의 신뢰구간 하한이 음수면 그대로 보고해도 되는가?

계산 전에 분모·단위·조건을 한 줄로 기록

In [ ]:
phat=2/100; se=np.sqrt(phat*(1-phat)/100); z=stats.norm.isf(.025)
wald=(phat-z*se,phat+z*se); wilson=wilson_interval(2,100)
print("Wald:",wald,"Wilson:",wilson)
assert wald[0]<0 and 0<=wilson[0]<wilson[1]<=1
scipy_ci=stats.binomtest(2,100).proportion_ci(method="wilson")
assert np.allclose(wilson,[scipy_ci.low,scipy_ci.high])

**조건 변경**: 성공수 2→0과 100으로 변경해 경계 확인

위 코드를 복사한 셀에서 한 조건씩 변경. 원본 셀은 보존

**선택 심화 · AI 검토**: 성공수2·시행수100·희소사건·신뢰수준·Wilson 또는 정확구간 요청

**결과 검토**: 음수 하한을 단순히 0으로 자르는 방법과 Wilson을 구분

예측·손계산·변경 결과·개념 설명·적용 한계 기록

In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


## S03. 원본 배터리 비율차·계란 모분산

**슬라이드 연결**: CE202_W11b_Inference4.pptx p9, 13, 17, 18, 19, 20

**자료 구분**: 원본 예제·수치 기반

**먼저 예측**: 비율차 구간과 모분산 구간의 단위·기준값은?

계산 전에 분모·단위·조건을 한 줄로 기록

In [ ]:
z=stats.norm.isf(.025) # 이 절의 95% 정규 임계값
pA,pB=10/500,12/1000; se_diff_p=np.sqrt(pA*(1-pA)/500+pB*(1-pB)/1000)
print("A−B:",pA-pB,"95% 정규근사:",(pA-pB)-z*se_diff_p,(pA-pB)+z*se_diff_p)
egg_g=np.array([45,47,44,48,46,45,47]); df=len(egg_g)-1; ss=np.sum((egg_g-egg_g.mean())**2)
variance_ci=(ss/stats.chi2.isf(.025,df),ss/stats.chi2.ppf(.025,df))
print("계란 표본분산:",egg_g.var(ddof=1),"모분산95%구간:",variance_ci)
assert np.isclose(egg_g.var(ddof=1),2) and variance_ci[0]<2<variance_ci[1]

**조건 변경**: 분산 구간에 쓰는 χ²의 양쪽 꼬리 순서를 바꾼 오류 검토

위 코드를 복사한 셀에서 한 조건씩 변경. 원본 셀은 보존

**선택 심화 · AI 검토**: A10/500·B12/1000, 계란g 자료7개·정규모집단 가정 명시

**결과 검토**: 0 포함이 두 비율의 같음을 증명하지 않음. 분산 단위는 g²

예측·손계산·변경 결과·개념 설명·적용 한계 기록

In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


## 주차 확인

대응 평균차·희소비율·분산 자유도 기록

입력 항목:

- `paired_mean_difference`: 교육 후−전 점수 차이의 평균 · 수치
- `wilson_lower_nonnegative`: 원본 2/100의 Wilson 95% 구간 하한이 0 이상인가? · Python `True`/`False`
- `egg_sample_variance`: 원본 계란 무게의 표본분산 (g²) · 수치
- `paired_data_proves_causality`: 같은 사람의 전후 자료라는 사실만으로 교육의 인과 효과가 입증되는가? · Python `True`/`False`

공개 확인용 항목. 결과값만 복사한 경우 학습 완료로 판단 불가. 분석 가정·자료 출처·추론 범위는 아래 기록과 함께 검토

In [ ]:
answers = {'paired_mean_difference': None, 'wilson_lower_nonnegative': None, 'egg_sample_variance': None, 'paired_data_proves_causality': None}


In [ ]:
def check_answers(values):
    expected = {'paired_mean_difference': 3.3, 'wilson_lower_nonnegative': True, 'egg_sample_variance': 2.0, 'paired_data_proves_causality': False}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## 원리 탐구·결과 해석 기록

1. 문제·자료·단위·모형과 가정
2. 실행 전 예측·작은 손계산과 근거
3. 기준 계산의 중간값·결과와 원리의 연결
4. 한 조건 변경 결과와 차이의 원인
5. 독립 검산·적용 조건·불확실성·한계
6. 자신의 말로 설명할 핵심 원리 한 가지

AI 활용은 선택. 사용 시 요청·의심한 출력·수정·검산 근거를 별도 기록. 글자 수 자동 채점 없음

In [ ]:
review_log = {
    "문제·자료·단위·가정": None,
    "예측·손계산 근거": None,
    "중간 계산·원리 연결": None,
    "조건 변경·차이의 원인": None,
    "검산·해석·적용 한계": None,
    "내 설명": None,
}
# 선택 활동: AI 활용 시 요청·의심한 출력·수정·검산 근거를 별도 기록
